Text data is everywhere in data science—whether as unstructured text files or as text columns within structured datasets. Developing strong skills for cleaning, processing, and analyzing text is essential for extracting insights and building effective models.

## Learning Objectives:

- Understand the importance of text preprocessing in NLP and data science workflows
- Learn how to clean and standardize raw text data
- Engineer features from text for downstream analysis and modeling
- Apply practical techniques to real-world datasets


## Dataset Case Study: My CHI My Future (MCMF) Programs in Chicago

<p style="display: flex; justify-content: center; align-items: center;">
    <img src="images/cropped-cropped-Final-Logo-1.png" alt="MCDC Logo" style="width: 20%; height: auto; margin-right: 2%;">
    <img src="images/mcmf_logo.png" alt="MCMF Logo" style="width: 20%; height: auto;">
</p>

This case study explores real-world data from the My CHI My Future (MCMF) youth programs in Chicago. The dataset was selected for the [MCDC](https://sites.northwestern.edu/mcdc/) practicum course (Spring 2023, Dr. Lizhen Shi) in partnership with [My CHI My Future](https://explore.mychimyfuture.org/) and community leader Stender Von Oehsen. The project aimed to address real community needs through data science, with results featured on the [MCDC Past Projects website](https://sites.northwestern.edu/mcdc/projects/).

**Context:**

- The MCMF initiative, led by the City of Chicago, connects youth to thousands of out-of-school programs, events, and resources across the city.
- The dataset includes detailed information about program names, descriptions, categories, dates, and locations, providing a rich resource for text analysis and feature engineering.
- Students worked with community partners to clean, analyze, and model this data, supporting efforts to improve youth engagement and access to opportunities.

This notebook demonstrates how to preprocess and analyze text data from a real civic dataset, preparing it for downstream modeling and actionable insights. This lesson was co-developed by **Dr. Lizhen Shi** and **Dr. Arend Kuyper** for the MCDC pathway courses (Summer 2025). 



## Notebook overview
This notebook walks through a full text-cleaning pipeline for the My CHI My Future programs data so learners can see the why behind each step. We'll:
1) Load the raw CSV and inspect structure/data quality.
2) Engineer date features (start year, duration) and prune to realistic records.
3) Focus on post-pandemic programs (2023-2025) and remove duration outliers.
4) Prepare category and text fields for downstream modeling.
5) Export a cleaned CSV used by later notebooks.

In [1]:
import pandas as pd
import numpy as np

## Loading the data

The dataset is publicly available on the [Chicago Data Portal](https://data.cityofchicago.org/Events/My-CHI-My-Future-Programs/w22p-bfyb/about_data). For this lesson, we dropped the columns that aren’t relevant to text analysis and kept only the text-focused fields (e.g., **program name** and **program description**) to simplify downstream cleaning and modeling.

Let's next load the dataset into a pandas DataFrame. 

#### How we'll load it
- Source: `datasets/My_CHI._My_Future._Programs_20250826.csv` (export from the Chicago Data Portal).
- Keep text-focused columns; numeric and geo fields are out-of-scope for this lesson.
- If you're low on memory, sample a subset first (e.g., `df.sample(5000, random_state=42)`) to iterate quickly.

In [2]:
df = pd.read_csv('datasets/My_CHI._My_Future._Programs_20250826.csv', low_memory=False)
df.head()

,Program ID,Program Name,Description,Category Name,Start Date,End Date,Start Time,End Time
0,138124,How to Finally Get Consistent with Healthy Eating,During this workshop youÕll identify the main ...,Academic Support,11/14/2022,11/14/2022,18:00,19:00
1,138610,Book Zoomies Activity Sheets,These activity sheets from Book Zoomies story ...,Music & Art.,11/25/2022,11/25/2022,12:00,13:00
2,137709,Book Zoomies Activity Sheets,These activity sheets from Book Zoomies story ...,Music & Art.,11/4/2022,11/4/2022,12:00,13:00
3,138702,"Dark Winds, Prey, Trickster and Reservation Do...","In honor of Native American Heritage Month, Je...",Music & Art.,11/29/2022,11/29/2022,18:00,18:45
4,121785,Homework Help: Virtual Teacher in the Library,Get homework help&nbsp;from a certified teache...,Music & Art.,5/24/2022,5/24/2022,NaN,NaN


Now let's examine the structure of our dataset to understand what columns we have available and their data types. This will help us identify the text fields we can analyze and any data quality issues we need to address.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 460531 entries, 0 to 460530
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   Program ID     460529 non-null  object
 1   Program Name   460529 non-null  object
 2   Description    460529 non-null  object
 3   Category Name  460526 non-null  object
 4   Start Date     460527 non-null  object
 5   End Date       460527 non-null  object
 6   Start Time     418345 non-null  object
 7   End Time       418299 non-null  object
dtypes: object(8)
memory usage: 28.1+ MB


#### What to look for in `info()`
- Text fields: confirm they are non-null strings and spot obvious missing descriptions.
- Date fields: ensure they come in as `object` so we can convert to datetime next.
- Row count: note the baseline size before any filtering so you can track drops.

## Data Cleaning

### Engineer date features (start year, duration) and prune to realistic records.

Next, we'll create additional columns from the date fields to support our analysis. We'll extract the year from the start date and calculate the duration of each program in days. This will help us understand temporal patterns and program length characteristics.

In [4]:
# Create new columns: Start Year and Duration
print("\nCREATING NEW DATE-BASED COLUMNS")
print("="*40)

# Convert date columns to datetime format
print("Converting date columns to datetime format...")
df['Start Date'] = pd.to_datetime(df['Start Date'], errors='coerce')
df['End Date'] = pd.to_datetime(df['End Date'], errors='coerce')

# Extract year from start date
df['Start_Year'] = df['Start Date'].dt.year
# Convert to nullable integer type to handle NaN values properly
df['Start_Year'] = df['Start_Year'].astype('Int64')
print(f"✓ Created 'Start_Year' column")

# Calculate duration between start and end dates (in days)
df['Duration_Days'] = (df['End Date'] - df['Start Date']).dt.days
print(f"✓ Created 'Duration_Days' column")

# Show sample of new columns
print(f"\nSample of new columns:")
df.head()


CREATING NEW DATE-BASED COLUMNS
Converting date columns to datetime format...
✓ Created 'Start_Year' column
✓ Created 'Duration_Days' column

Sample of new columns:


,Program ID,Program Name,Description,Category Name,Start Date,End Date,Start Time,End Time,Start_Year,Duration_Days
0,138124,How to Finally Get Consistent with Healthy Eating,During this workshop youÕll identify the main ...,Academic Support,2022-11-14,2022-11-14,18:00,19:00,2022,0.0
1,138610,Book Zoomies Activity Sheets,These activity sheets from Book Zoomies story ...,Music & Art.,2022-11-25,2022-11-25,12:00,13:00,2022,0.0
2,137709,Book Zoomies Activity Sheets,These activity sheets from Book Zoomies story ...,Music & Art.,2022-11-04,2022-11-04,12:00,13:00,2022,0.0
3,138702,"Dark Winds, Prey, Trickster and Reservation Do...","In honor of Native American Heritage Month, Je...",Music & Art.,2022-11-29,2022-11-29,18:00,18:45,2022,0.0
4,121785,Homework Help: Virtual Teacher in the Library,Get homework help&nbsp;from a certified teache...,Music & Art.,2022-05-24,2022-05-24,NaN,NaN,2022,0.0


Let's examine the distribution of programs across different years and analyze the quality of our date data. This will help us understand temporal patterns and identify any data quality issues that need to be addressed.

Before filtering, we profile the new `Start_Year` and `Duration_Days` columns to see how reliable the dates are and where obvious issues (missing/negative/too long) show up.

In [5]:
# Display statistics about the new columns
print(f"\nStart Year distribution:")
year_counts = df['Start_Year'].value_counts().sort_index()
print(year_counts.head(10))

print(f"\nDuration statistics:")
duration_stats = df['Duration_Days'].describe()
print(duration_stats)

# Check for any issues with the data
print(f"\nData quality check:")
print(f"- Programs with missing Start Date: {df['Start Date'].isna().sum():,}")
print(f"- Programs with missing End Date: {df['End Date'].isna().sum():,}")
print(f"- Programs with negative duration: {(df['Duration_Days'] < 0).sum():,}")
print(f"- Programs with very long duration (>365 days): {(df['Duration_Days'] > 365).sum():,}")


Start Year distribution:
Start_Year
2020       208
2021     46987
2022     93328
2023    102343
2024    136620
2025     81024
2026        12
2054         3
Name: count, dtype: Int64

Duration statistics:
count    460525.000000
mean         34.023378
std         137.853833
min      -10577.000000
25%           0.000000
50%           0.000000
75%          63.000000
max       10227.000000
Name: Duration_Days, dtype: float64

Data quality check:
- Programs with missing Start Date: 6
- Programs with missing End Date: 6
- Programs with negative duration: 3
- Programs with very long duration (>365 days): 875


###  Data Cleaning: Focus on post-pandemic programs (2023-2025) and remove duration outliers.

#### Post-pandemic Programs

Based on our year distribution analysis, we can see that the program counts for 2020-2022 were significantly affected by the COVID-19 pandemic. From 2023 onwards, youth programs appear to have returned to normal levels.

For our text analysis, we'll focus on programs from **2023-2025** as these represent:
- Post-pandemic program offerings that reflect current trends
- More recent and relevant program descriptions
- Data that is most applicable to current programming decisions

We'll also remove any programs with start years beyond 2025, as these are likely data entry errors given our current date.

In [6]:
# Filter the dataset to include only programs from 2023-2025
print("FILTERING DATASET BY YEAR")
print("="*30)

# Show original dataset size
print(f"Original dataset size: {len(df):,} programs")

# Filter for programs starting in 2023, 2024, or 2025
df_filtered = df[df['Start_Year'].isin([2023, 2024, 2025])].copy()

print(f"Filtered dataset size: {len(df_filtered):,} programs")
print(f"Programs removed: {len(df) - len(df_filtered):,}")
print(f"Percentage retained: {(len(df_filtered) / len(df) * 100):.1f}%")

# Show year distribution in filtered dataset
print(f"\nYear distribution in filtered dataset:")
filtered_year_counts = df_filtered['Start_Year'].value_counts().sort_index()
for year, count in filtered_year_counts.items():
    percentage = (count / len(df_filtered)) * 100
    print(f"  {year}: {count:,} programs ({percentage:.1f}%)")

FILTERING DATASET BY YEAR
Original dataset size: 460,531 programs
Filtered dataset size: 319,987 programs
Programs removed: 140,544
Percentage retained: 69.5%

Year distribution in filtered dataset:
  2023: 102,343 programs (32.0%)
  2024: 136,620 programs (42.7%)
  2025: 81,024 programs (25.3%)


#### Checkpoint: post-year filter
Capture the retained row count and year distribution shown above; you can log this number in reports so downstream notebooks know the cohort size they are using.

#### Remove Duration Outliers

Now let's clean up the data based on program duration. We'll remove programs with unrealistic durations that could indicate data quality issues:
- Programs with negative durations (end date before start date)
- Programs with extremely long durations (over 1 year) which may be data entry errors
- Zero duration programs represent single-day programs or events that last only a couple hours within the same day

Let's create a `Duration_Group` column to better categorize programs:

* Same day (0 days) - for events/programs within a single day
* 1 day - for programs spanning exactly one day
* 2-7 days - for short multi-day programs
* 1-4 weeks, 1-3 months, 3-6 months, 6-12 months - for longer programs

This will help ensure our text analysis is based on high-quality, realistic program data.

In [7]:
# Clean up data based on Duration_Days
print("DURATION-BASED DATA CLEANUP")
print("="*35)

print(f"Before duration cleanup: {len(df_filtered):,} programs")

# Check current duration issues
print(f"\nDuration quality issues in filtered data:")
negative_duration = (df_filtered['Duration_Days'] < 0).sum()
very_long_duration = (df_filtered['Duration_Days'] > 365).sum()
zero_duration = (df_filtered['Duration_Days'] == 0).sum()
missing_duration = df_filtered['Duration_Days'].isna().sum()

print(f"- Negative duration: {negative_duration:,} programs")
print(f"- Very long duration (>365 days): {very_long_duration:,} programs") 
print(f"- Zero duration (same day programs): {zero_duration:,} programs")
print(f"- Missing duration: {missing_duration:,} programs")

# Apply duration-based filters
print(f"\nApplying duration filters...")

# Remove programs with negative durations
df_filtered = df_filtered[df_filtered['Duration_Days'] >= 0]
print(f"✓ Removed programs with negative duration")

# Remove programs with extremely long durations (>365 days)
df_filtered = df_filtered[df_filtered['Duration_Days'] <= 365]
print(f"✓ Removed programs with duration >365 days")

# Remove programs with missing duration data
df_filtered = df_filtered[df_filtered['Duration_Days'].notna()]
print(f"✓ Removed programs with missing duration")

print(f"\nAfter duration cleanup: {len(df_filtered):,} programs")
print(f"Programs removed in cleanup: {len(df[df['Start_Year'].isin([2023, 2024, 2025])]) - len(df_filtered):,}")

# Show final duration statistics
print(f"\nFinal duration statistics:")
final_duration_stats = df_filtered['Duration_Days'].describe()
print(final_duration_stats)

# Show duration distribution
print(f"\nDuration distribution (ranges):")
duration_ranges = pd.cut(df_filtered['Duration_Days'], 
                        bins=[-0.1, 0, 1, 7, 30, 90, 180, 365], 
                        labels=['Same day', '1 day', '2-7 days', '1-4 weeks', '1-3 months', '3-6 months', '6-12 months'],
                        include_lowest=True)
duration_counts = duration_ranges.value_counts().sort_index()
for range_label, count in duration_counts.items():
    percentage = (count / len(df_filtered)) * 100
    print(f"  {range_label}: {count:,} programs ({percentage:.1f}%)")

DURATION-BASED DATA CLEANUP
Before duration cleanup: 319,987 programs

Duration quality issues in filtered data:
- Negative duration: 0 programs
- Very long duration (>365 days): 616 programs
- Zero duration (same day programs): 179,038 programs
- Missing duration: 0 programs

Applying duration filters...
✓ Removed programs with negative duration
✓ Removed programs with duration >365 days
✓ Removed programs with missing duration

After duration cleanup: 319,371 programs
Programs removed in cleanup: 616

Final duration statistics:
count    319371.000000
mean         27.499645
std          36.845430
min           0.000000
25%           0.000000
50%           0.000000
75%          63.000000
max         365.000000
Name: Duration_Days, dtype: float64

Duration distribution (ranges):
  Same day: 179,038 programs (56.1%)
  1 day: 167 programs (0.1%)
  2-7 days: 3,597 programs (1.1%)
  1-4 weeks: 6,615 programs (2.1%)
  1-3 months: 115,562 programs (36.2%)
  3-6 months: 13,015 programs (4.1%)


#### Why these duration filters
- Negative durations usually mean swapped start/end dates.
- >365-day spans look like data entry errors for youth programs.
- Dropping NAs keeps later grouping math simple.
If your use-case needs camps that run all year, loosen these bounds here.

#### Bucket program durations

We convert raw `Duration_Days` into a small set of interpretable bins (same day, 1 day, 2-7 days, etc.). These buckets make it easier to segment programs and later join duration as a categorical feature.

In [8]:
# Create Duration_Group column as a separate feature for analysis
print("CREATING DURATION_GROUP COLUMN")
print("="*35)

# Create categorical column based on duration ranges
duration_ranges = pd.cut(df_filtered['Duration_Days'], 
                        bins=[-0.1, 0, 1, 7, 30, 90, 180, 365], 
                        labels=['Same day', '1 day', '2-7 days', '1-4 weeks', '1-3 months', '3-6 months', '6-12 months'],
                        include_lowest=True)

df_filtered['Duration_Group'] = duration_ranges
print(f"✓ Created 'Duration_Group' column with {df_filtered['Duration_Group'].nunique()} categories")

df_filtered.head()


print(f"\n✓ Duration_Group column ready for further analysis and grouping!")

CREATING DURATION_GROUP COLUMN
✓ Created 'Duration_Group' column with 7 categories

✓ Duration_Group column ready for further analysis and grouping!


### EDA: Check category balance
Category counts help us spot label imbalance (rare vs. common program types) before we build any models or sampling strategies.

In [9]:
# Analyze Category Name distribution in the filtered dataset
print("CATEGORY NAME ANALYSIS")
print("="*30)

# Compute value counts for Category Name
category_counts = df_filtered['Category Name'].value_counts()
print(f"Total unique categories: {df_filtered['Category Name'].nunique()}")
print(f"Total programs: {len(df_filtered):,}")

print(f"\nCategory distribution:")
print(f"Top 20 most common categories:")
for i, (category, count) in enumerate(category_counts.head(20).items(), 1):
    percentage = (count / len(df_filtered)) * 100
    print(f"  {i:2d}. {category}: {count:,} programs ({percentage:.1f}%)")

# Show categories with very few programs
print(f"\nCategories with fewer than 100 programs:")
rare_categories = category_counts[category_counts < 100]
print(f"Number of rare categories: {len(rare_categories)}")
for category, count in rare_categories.items():
    print(f"  - {category}: {count} programs")

# Show summary statistics
print(f"\nCategory statistics:")
print(f"- Most popular category: {category_counts.index[0]} ({category_counts.iloc[0]:,} programs)")
print(f"- Average programs per category: {category_counts.mean():.1f}")
print(f"- Median programs per category: {category_counts.median():.1f}")
print(f"- Categories with >1000 programs: {(category_counts > 1000).sum()}")
print(f"- Categories with >500 programs: {(category_counts > 500).sum()}")
print(f"- Categories with >100 programs: {(category_counts > 100).sum()}")

CATEGORY NAME ANALYSIS
Total unique categories: 21
Total programs: 319,371

Category distribution:
Top 20 most common categories:
   1. Sports + Wellness.: 122,148 programs (38.2%)
   2. Music & Art.: 105,679 programs (33.1%)
   3. Reading & Writing.: 34,538 programs (10.8%)
   4. Academic Support: 13,007 programs (4.1%)
   5. Science: 9,442 programs (3.0%)
   6. Computers.: 9,297 programs (2.9%)
   7. Building & Fixing Things: 6,617 programs (2.1%)
   8. Helping Your Community.: 3,223 programs (1.0%)
   9. Performance.: 2,624 programs (0.8%)
  10. Nature.: 2,569 programs (0.8%)
  11. Managing Money.: 2,177 programs (0.7%)
  12. Healthcare: 2,161 programs (0.7%)
  13. Food.: 1,767 programs (0.6%)
  14. Work + Career: 1,665 programs (0.5%)
  15. Digital Media.: 1,557 programs (0.5%)
  16. Social Studies: 499 programs (0.2%)
  17. Teaching: 167 programs (0.1%)
  18. Customer/Human Service: 94 programs (0.0%)
  19. Math: 60 programs (0.0%)
  20. Law: 54 programs (0.0%)

Categories with fe

#### How to use category balance
- Imbalanced labels influence both clustering visuals and any supervised baselines.
- Consider collapsing very rare categories or reweighting later if you plan to train classifiers.
- Keep a note of the top categories; later notebooks filter to them.

## Text Analysis Phase

Now that we have a clean, filtered dataset with 2023-2025 programs, let's dive into the core text analysis. We'll focus on analyzing the **Program Name** and **Description** fields to extract meaningful insights.

### Text Preprocessing & Cleaning
- Clean HTML tags and special characters from descriptions
- Standardize text format (lowercase, remove punctuation)
- Remove stop words and short words
- Handle common text issues


### Peek at raw text
A quick look at a few names and descriptions grounds the cleaning choices - we want to see real noise like HTML tags, boilerplate, or contact info before writing a cleaning function.

In [10]:
# 1. TEXT PREPROCESSING & CLEANING
print("TEXT PREPROCESSING & CLEANING")
print("="*40)

import re
import string
from collections import Counter

# Sample text data to understand what we're working with
print(f"\n Sample Program Names:")
sample_names = df_filtered['Program Name'].dropna().head()
for i, name in enumerate(sample_names, 1):
    print(f"  {i}. {name}")

print(f"\n Sample Descriptions (first 500 chars):")
sample_descriptions = df_filtered['Description'].dropna().head(3)
for i, desc in enumerate(sample_descriptions, 1):
    print(f"  {i}. {str(desc)[:500]}...")

print("\n" + "="*50)

TEXT PREPROCESSING & CLEANING

 Sample Program Names:
  1. Homework Help: Virtual Teacher in the Library
  2. Art of Storytelling Learning Circle
  3. Learn American Sign Language
  4. Learn American Sign Language
  5. The Harold Washington Experience: Historians Reflect on His Impact

 Sample Descriptions (first 500 chars):
  1. Get homework help from a certified teacher and learn about library resources with our expert librarians. Join other students and their families in this after-school learning program. Attend the whole session or drop in with a quick question. For students in grades K to 12. How to Attend: This event takes place on Zoom. Register by 4 p.m. the day of the event. Chicago Public Library cannot collect personal information online from kids 0 to 13. A parent or guardian's email address must be used to ...
  2. What makes for a good story? In this course developed by Pixar, learn how a story's structure, characters, visuals, and cinematography contribute to a story. W

### Build a reusable cleaning function
We normalize text by lowercasing, stripping HTML/URLs/emails/numbers, removing stopwords, and dropping very short tokens. This creates `Program_Name_Clean` and `Description_Clean` fields ready for vectorization.

#### Setup note for stopwords
`nltk.download('stopwords')` only needs to run once per environment. If you're on a locked-down machine, you can vendor the stopword list manually instead of downloading at runtime.

In [11]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

# Create text cleaning function
def clean_text(text, min_length=3):
    """Clean and normalize text for analysis"""
    if pd.isna(text) or text == '':
        return ''
    
    # Convert to string and lowercase
    text = str(text).lower()
    
    # Remove HTML tags (basic cleaning)
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # Remove URLs
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', ' ', text)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)
    
    # Remove numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace and punctuation
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Remove Stopwords - Common words that add little meaning (e.g., 'the', 'is', 'in')
    words = [word for word in text.split() 
             if len(word) >= min_length and word not in stop_words]
    
    return ' '.join(words)

# Apply text cleaning
print(" Cleaning text fields...")
df_filtered['Program_Name_Clean'] = df_filtered['Program Name'].apply(clean_text)
df_filtered['Description_Clean'] = df_filtered['Description'].apply(clean_text)

# Show before/after examples
print(f"\n Before/After Cleaning Examples:")
for i in range(3):
    if i < len(df_filtered):
        print(f"\nExample {i+1}:")
        original_name = df_filtered.iloc[i]['Program Name']
        clean_name = df_filtered.iloc[i]['Program_Name_Clean']
        print(f"  Original Name: {original_name}")
        print(f"  Clean Name: {clean_name}")
        
        original_desc = str(df_filtered.iloc[i]['Description'])[:150] + "..."
        clean_desc = df_filtered.iloc[i]['Description_Clean'][:150] + "..."
        print(f"  Original Desc: {original_desc}")
        print(f"  Clean Desc: {clean_desc}")
        print("-" * 60)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/R8C6026/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


 Cleaning text fields...

 Before/After Cleaning Examples:

Example 1:
  Original Name: Homework Help: Virtual Teacher in the Library
  Clean Name: homework help virtual teacher library
  Original Desc: Get homework help from a certified teacher and learn about library resources with our expert librarians. Join other students and their families in thi...
  Clean Desc: get homework help certified teacher learn library resources expert librarians join students families afterschool learning program attend whole session...
------------------------------------------------------------

Example 2:
  Original Name: Art of Storytelling Learning Circle
  Clean Name: art storytelling learning circle
  Original Desc: What makes for a good story? In this course developed by Pixar, learn how a story's structure, characters, visuals, and cinematography contribute to a...
  Clean Desc: makes good story course developed pixar learn storys structure characters visuals cinematography contribute story exp

### Add simple length features
Word counts for names and descriptions (`Name_Word_Count`, `Desc_Word_Count`) give a quick quality check and can serve as lightweight features or filters.

In [12]:
# add word count columns for program name and description
s = df_filtered["Program_Name_Clean"].astype("string")
df_filtered["Name_Word_Count"] = s.str.split().str.len()

s= df_filtered["Description_Clean"].astype("string")
df_filtered["Desc_Word_Count"] = s.str.split().str.len()
df_filtered.head()

,Program ID,Program Name,Description,Category Name,Start Date,End Date,Start Time,End Time,Start_Year,Duration_Days,Duration_Group,Program_Name_Clean,Description_Clean,Name_Word_Count,Desc_Word_Count
6,231612,Homework Help: Virtual Teacher in the Library,Get homework help from a certified teacher and...,Music & Art.,2025-02-06,2025-02-06,17:00,19:00,2025,0.0,Same day,homework help virtual teacher library,get homework help certified teacher learn libr...,5,109
17,220338,Art of Storytelling Learning Circle,What makes for a good story? In this course de...,Academic Support,2024-10-02,2024-10-02,14:00,15:30,2024,0.0,Same day,art storytelling learning circle,makes good story course developed pixar learn ...,4,54
19,156338,Learn American Sign Language,Want to learn or brush up on your American Sig...,Academic Support,2023-04-25,2023-04-25,15:30,16:30,2023,0.0,Same day,learn american sign language,want learn brush american sign language skills...,4,81
20,211006,Learn American Sign Language,Want to learn or brush up on your American Sig...,Academic Support,2024-08-27,2024-08-27,15:30,16:15,2024,0.0,Same day,learn american sign language,want learn brush american sign language skills...,4,111
22,199268,The Harold Washington Experience: Historians R...,"In April 1983, Harold Washington made history ...",Academic Support,2024-04-25,2024-04-25,18:00,19:00,2024,0.0,Same day,harold washington experience historians reflec...,april harold washington made history becoming ...,6,191


### Quick exploratory text stats
Basic length stats and top tokens show what dominates the corpus and whether extra cleaning or rebalancing is needed.

In [13]:
# 2. EXPLORATORY TEXT ANALYSIS
print("\n" + "="*50)
print("EXPLORATORY TEXT ANALYSIS")
print("="*50)

# Text length analysis
print("\nProgram Names (cleaned) Statistics:")

name_stats = df_filtered['Name_Word_Count'].describe()
print(f"  Mean: {name_stats['mean']:.1f} words")
print(f"  Median: {name_stats['50%']:.1f} words")
print(f"  Range: {name_stats['min']:.0f} - {name_stats['max']:.0f} words")

# Word frequency analysis
print(f"\n Word Frequency Analysis:")

# Most common words in program names
name_words = []
for name in df_filtered['Program_Name_Clean'].dropna():
    name_words.extend(name.split())

name_word_counts = Counter(name_words)
print(f"\nTop 15 words in Program Names:")
for word, count in name_word_counts.most_common(15):
    print(f"  {word}: {count:,} times")


print("\nProgram Description (cleaned) Statistics:")

desc_stats = df_filtered['Desc_Word_Count'].describe()
print(f"  Mean: {desc_stats['mean']:.1f} words")
print(f"  Median: {desc_stats['50%']:.1f} words")
print(f"  Range: {desc_stats['min']:.0f} - {desc_stats['max']:.0f} words")

# Most common words in descriptions
desc_words = []
for desc in df_filtered['Description_Clean'].dropna():
    desc_words.extend(desc.split())

desc_word_counts = Counter(desc_words)
print(f"\nTop 15 words in Descriptions:")
for word, count in desc_word_counts.most_common(15):
    print(f"  {word}: {count:,} times")

# Text quality assessment
print(f"\n Text Quality Assessment based on desc_word_count:")
very_long_descs = (df_filtered['Desc_Word_Count'] > 500).sum()
print(f"Very long descriptions (>500 words): {very_long_descs:,} ({very_long_descs/len(df_filtered)*100:.1f}%)")


print("\n Text preprocessing and exploratory analysis complete!")


EXPLORATORY TEXT ANALYSIS

Program Names (cleaned) Statistics:
  Mean: 3.8 words
  Median: 3.0 words
  Range: 0 - 27 words

 Word Frequency Analysis:

Top 15 words in Program Names:
  time: 22,623 times
  open: 20,845 times
  story: 19,232 times
  club: 17,661 times
  gymnastics: 17,323 times
  park: 14,920 times
  ice: 14,649 times
  swim: 12,533 times
  mcfetridge: 11,853 times
  basketball: 11,419 times
  film: 11,377 times
  camp: 11,281 times
  screening: 11,069 times
  teen: 9,893 times
  skating: 9,811 times

Program Description (cleaned) Statistics:
  Mean: 45.9 words
  Median: 38.0 words
  Range: 1 - 757 words

Top 15 words in Descriptions:
  event: 326,678 times
  days: 216,090 times
  accessibility: 191,723 times
  email: 180,708 times
  must: 174,776 times
  please: 170,029 times
  library: 146,332 times
  week: 138,923 times
  least: 136,703 times
  call: 135,389 times
  assistance: 131,891 times
  language: 131,135 times
  made: 130,858 times
  business: 130,181 times
  

### Add a Domain-Specific Stoplist  

As seen in the results above, program descriptions often contain repeated boilerplate phrases (e.g., *“registration required”*, *“free”*, *“north side”*, or phone numbers).  
To reduce noise and improve feature quality, create a custom stoplist called `domain_stop_words` and apply it to further clean the descriptions.


In [14]:
# add domain stop words
domain_stop_words = {'event', 'attend', 'attending', 'need', 'using', 'fun', 'email', 'must', 'please', 'least', 'call', 'made', 'business', 'days', 'program', 'programs', 'requests', 'request', 'skills', 'chicago', 'kids', 'children', 'participants', 'service', 'services', 'community', 'youth', 'students', 'learn', 'learning', 'support', 'provide', 'includes', 'including', 'activities', 'activity', 'event','events','day','days','week','weeks','email','call','please','must','sign','language','assistance','accessibility',
    'contact','information','phone','required','register','registration','free',
    'learn','session','provide','available','time','date','location','virtual','in-person','online','join','participate','open','public','workshop','workshops','class','classes','course','courses','club','clubs','team','teams','meetup','meetups','series','series','month','months','year','years','center','centers','center.','centers.','school','schools','camp','camps','campus','facilities','facility','facilities'}

# Further clean the Description_Clean column  by removing domain-specific stop words, months, and days
def remove_domain_stopwords(text):
    if pd.isna(text) or text == '':
        return ''
    words = text.split()
    filtered_words = [word for word in words if word not in domain_stop_words]
    return ' '.join(filtered_words)
df_filtered['Description_Clean'] = df_filtered['Description_Clean'].apply(remove_domain_stopwords)
print(f"✓ Removed domain-specific stop words from 'Description_Clean' column")

# Most common words in descriptions
desc_words = []
for desc in df_filtered['Description_Clean'].dropna():
    desc_words.extend(desc.split())

desc_word_counts = Counter(desc_words)
print(f"\nTop 15 words in Descriptions:")
for word, count in desc_word_counts.most_common(15):
    print(f"  {word}: {count:,} times")


✓ Removed domain-specific stop words from 'Description_Clean' column

Top 15 words in Descriptions:
  library: 146,332 times
  interpretation: 127,900 times
  accommodations: 127,305 times
  play: 78,704 times
  check: 69,037 times
  cpl: 66,110 times
  place: 60,163 times
  encouraged: 57,749 times
  takes: 57,578 times
  person: 56,296 times
  games: 53,269 times
  level: 53,265 times
  park: 51,211 times
  visiting: 50,847 times
  strongly: 49,549 times


#### Tuning the domain stoplist
Iterate on this list as you explore: copy/paste high-frequency boilerplate phrases you still see in top tokens or word clouds. Save a dated version so teammates know which run produced which cleaned file.

In [16]:
df_filtered.groupby('Category Name').size()

Category Name
Academic Support             13007
Building & Fixing Things      6617
Computers.                    9297
Customer/Human Service          94
Digital Media.                1557
Food.                         1767
Healthcare                    2161
Helping Your Community.       3223
Law                             54
Managing Money.               2177
Math                            60
Music & Art.                105679
Nature.                       2569
Performance.                  2624
Reading & Writing.           34538
Science                       9442
Social Studies                 499
Sports + Wellness.          122148
Teaching                       167
Transportation                  26
Work + Career                 1665
dtype: int64

### Save the cleaned dataset
We export the filtered, cleaned records to `datasets/data_cleaned_20250826.csv` so downstream notebooks can focus on modeling without repeating the prep steps.

In [17]:
df_filtered.to_csv('datasets/data_cleaned_20250826.csv', index=False)

In [20]:
df_filtered.shape, df.shape,len(df_filtered)/len(df)

((319371, 15), (460531, 10), 0.6934842605600926)

In [23]:
df_filtered['Category Name'].value_counts().index.to_list()

['Sports + Wellness.',
 'Music & Art.',
 'Reading & Writing.',
 'Academic Support',
 'Science',
 'Computers.',
 'Building & Fixing Things',
 'Helping Your Community.',
 'Performance.',
 'Nature.',
 'Managing Money.',
 'Healthcare',
 'Food.',
 'Work + Career',
 'Digital Media.',
 'Social Studies',
 'Teaching',
 'Customer/Human Service',
 'Math',
 'Law',
 'Transportation']

In [29]:
df_filtered.sample(1).to_dict()

{'Program ID': {59769: '241497'},
 'Program Name': {59769: 'Flag Football at Addams'},
 'Description': {59769: 'The proper fundamentals and techniques of football will be taught in this activity. Players will receive instruction in passing, catching and skills training while also learning offensive plays, defensive plays and game strategy. Days of the week: Monday'},
 'Category Name': {59769: 'Sports + Wellness.'},
 'Start Date': {59769: Timestamp('2025-03-31 00:00:00')},
 'End Date': {59769: Timestamp('2025-05-19 00:00:00')},
 'Start Time': {59769: '18:00'},
 'End Time': {59769: '18:45'},
 'Start_Year': {59769: 2025},
 'Duration_Days': {59769: 49.0},
 'Duration_Group': {59769: '1-3 months'},
 'Program_Name_Clean': {59769: 'flag football addams'},
 'Description_Clean': {59769: 'proper fundamentals techniques football taught players receive instruction passing catching training also offensive plays defensive plays game strategy monday'},
 'Name_Word_Count': {59769: 3},
 'Desc_Word_Count

## Wrap-up and next steps
You now have a cleaned, filtered, and tokenized dataset saved to `datasets/data_cleaned_20250826.csv`.
- Hand off to clustering/classification notebooks (`2_text_analysis_category_clustering.ipynb`, `3_text_analysis_category_quality.ipynb`).
- If results look off later, revisit the checkpoints above (year filter size, duration thresholds, stoplist contents) and adjust with notes so analyses stay reproducible.